# Stock Price Prediction using Machine Learning (ML)

### Using scikit-learn library for ML

In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Any
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [7]:
# Function: Get Historical Data
def get_stock_data(ticker:str, start_date:str, end_date:str) -> Any:
    data = yf.download(ticker, start = start_date, end = end_date)
    return data

In [8]:
# Function: Feature Engineering

def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # Calculate daily return
    df['Return'] = df['Close'].pct_change()

    # Calculate moving averages
    df['SMA_5'] = df['Close'].rolling(window = 5).mean()
    df['SMA_10'] = df['Close'].rolling(window = 10).mean()

    # Calculate Volatility: 5 day standard deviation from closing prices
    df['Volatility_5'] = df['Close'].rolling(window=5).std()

    # Create lag features: Closing price from 1, 2, 3 days ago
    # If you increase lag features also update prepare data
    for i in range(1,4):
        df[f"Close_lag{i}"] = df["Close"].shift(i) # Shift function adds NaN to first row in Column Close_lag[i]
    
    
    # Target: This is what we want to predict: Tomorrow's closing price
    df['Target'] = df['Close'].shift(-1)

    print("\nProcessed Date:\n")
    print(df.head(15))

    # Remove rows with missing values
    df.dropna(inplace = True)

    return df
    

In [11]:
# Function Prepare data for modeling

def prepare_data(df:pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    #Define a list of columns that are going to be forwarded to the ML model

    features = [
        'Close_lag1',
        'Close_lag2',
        'Close_lag3',
        'SMA_5',
        'SMA_10',
        'Volatility_5',
        'Return'
    ]

    X = df[features]
    y = df['Target']

    return X, y
        

In [12]:
# Splitting training and testing data
def train_test_split_ts(X: pd.DataFrame, y:pd.Series, split: float = 0.8) -> Tuple:
    split_idx = int(len(X) * split)

    return(
        X.iloc[:split_idx],
        X.iloc[spit_idx:],
        y.iloc[:split_idx],
        y.iloc[spit_idx:],
    )

In [ ]:
# Model Training
def train_model(X_train: pd.DataFrame, y_train: pd.Series, model_type: str = "lr") -> Any:
    